# Practice 03 — Channel & Paid Analysis
**Context:** The paid media team wants to understand how paid traffic compares to organic across the funnel, and whether their spend is generating quality activations.

This is the kind of deepdive you'd do when the Adobe Analytics dashboard isn't enough.

---
**Reference notebooks:** `course/03_chart_types.ipynb`, `course/04_subplots_layouts.ipynb`, `course/06_statistical_plots.ipynb`

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('data/raw/funnel_data.csv', parse_dates=['date'])
print(df['channel'].unique())
print(df.shape)

---
## Exercise 1 — CVR distribution: paid vs organic

Are paid users converting at a statistically different rate than organic?

Build a boxplot comparing CVR (visit → activation) distributions for all channels.

Requirements:
- Compute daily CVR per channel per day
- Boxplot with `notch=True` and `patch_artist=True`
- Each channel a different color
- Add a horizontal dashed line at the overall mean CVR

💡 Hint: `cvr = df.groupby(['date','channel']).apply(lambda g: g['activacion_tarjeta'].sum() / g['visita_landing'].sum() * 100)`

In [ ]:

# Exercise 1 — CVR distribution: paid vs organic

cvr = df.groupby(['date', 'channel']).apply(
    lambda g: g['activacion_tarjeta'].sum() / g['visita_landing'].sum() * 100
).reset_index(name='cvr')

channels = cvr['channel'].unique()
colors   = ['#4361ee', '#f72585', '#7209b7', '#3a0ca3', '#4cc9f0']
data_per_channel = [cvr.loc[cvr['channel'] == ch, 'cvr'].values for ch in channels]

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(
    data_per_channel,
    labels=channels,
    notch=True,
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

mean_cvr = cvr['cvr'].mean()
ax.axhline(mean_cvr, linestyle='--', color='gray', linewidth=1.2,
           label=f'Overall mean CVR: {mean_cvr:.1f}%')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('CVR Distribution by Channel (Visit → Activation)', fontsize=14, fontweight='bold')
ax.set_ylabel('CVR (%)')
ax.set_xlabel('Channel')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


---
## Exercise 2 — Stacked area: volume by channel over time

Show how each channel contributes to total daily sessions over the 90-day period.

Requirements:
- `ax.stackplot()` with one color per channel
- Legend outside the chart
- Weekend shading: use `axvspan` to shade every Saturday-Sunday pair

💡 Hint to find weekends:
```python
for date in pivot.index:
    if date.weekday() >= 5:  # 5=Saturday, 6=Sunday
        ax.axvspan(...)
```

In [ ]:

# Exercise 2 — Stacked area: volume by channel over time

daily_channel = df.groupby(['date', 'channel'])['visita_landing'].sum().unstack(fill_value=0)
channels_ordered = daily_channel.columns.tolist()
colors = ['#4361ee', '#f72585', '#7209b7', '#3a0ca3', '#4cc9f0']

fig, ax = plt.subplots(figsize=(12, 5))

ax.stackplot(
    daily_channel.index,
    [daily_channel[ch] for ch in channels_ordered],
    labels=channels_ordered,
    colors=colors[:len(channels_ordered)],
    alpha=0.85,
)

# Weekend shading
for date in daily_channel.index:
    if date.weekday() == 5:  # Saturday
        ax.axvspan(date, date + pd.Timedelta(days=2), color='gray', alpha=0.10, linewidth=0)

ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, title='Channel')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('Daily Sessions by Channel', fontsize=14, fontweight='bold')
ax.set_ylabel('Sessions')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()


---
## Exercise 3 — Scatter: does more paid traffic mean more activations?

Build a scatter plot with paid sessions on X and paid activations on Y.

Requirements:
- One dot per day
- Add a linear regression line with `np.polyfit`
- Show the R² value in the chart using `ax.text()` or `ax.annotate()`
- Color dots by month (January=blue, February=orange, March=green)

💡 Hint: `r_squared = np.corrcoef(x, y)[0,1]**2`

In [ ]:

# Exercise 3 — Scatter: paid sessions vs paid activations

paid = df[df['channel'] == 'paid'].groupby('date').agg(
    sessions=('visita_landing', 'sum'),
    activations=('activacion_tarjeta', 'sum'),
).reset_index()
paid['month'] = paid['date'].dt.month

month_colors = {1: '#4361ee', 2: '#f72585', 3: '#06d6a0'}
month_labels = {1: 'January', 2: 'February', 3: 'March'}

fig, ax = plt.subplots(figsize=(8, 6))

for month, grp in paid.groupby('month'):
    ax.scatter(grp['sessions'], grp['activations'],
               color=month_colors[month], label=month_labels[month],
               s=50, alpha=0.8, edgecolors='white', linewidths=0.5)

# Regression line
x, y = paid['sessions'].values, paid['activations'].values
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, m * x_line + b, color='#333', linewidth=1.5, linestyle='--')

r_squared = np.corrcoef(x, y)[0, 1] ** 2
ax.text(0.05, 0.93, f'R² = {r_squared:.3f}', transform=ax.transAxes,
        fontsize=11, color='#333')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('Paid Sessions vs Paid Activations (daily)', fontsize=14, fontweight='bold')
ax.set_xlabel('Paid Sessions')
ax.set_ylabel('Paid Activations')
ax.legend(frameon=False, title='Month')
plt.tight_layout()
plt.show()


---
## Exercise 4 — KDE: OTP completion rate by channel

OTP is a common drop-off point. Compare the distribution of OTP completion rates across channels using KDE curves.

Requirements:
- OTP rate = `otp / inicio_solicitud * 100` per day per channel
- One KDE curve per channel, different color
- Fill under each curve with low alpha
- Add a vertical line at the median of each channel

💡 Hint: `from scipy import stats; kde = stats.gaussian_kde(values)`

In [ ]:

# Exercise 4 — KDE: OTP completion rate by channel

otp_rate = df.groupby(['date', 'channel']).apply(
    lambda g: g['otp'].sum() / g['inicio_solicitud'].sum() * 100
).reset_index(name='otp_rate')

channels = otp_rate['channel'].unique()
colors   = ['#4361ee', '#f72585', '#7209b7', '#3a0ca3', '#4cc9f0']

fig, ax = plt.subplots(figsize=(10, 6))

for ch, color in zip(channels, colors):
    values = otp_rate.loc[otp_rate['channel'] == ch, 'otp_rate'].dropna().values
    kde = stats.gaussian_kde(values)
    x_range = np.linspace(values.min() - 2, values.max() + 2, 300)
    y_kde = kde(x_range)

    ax.plot(x_range, y_kde, color=color, linewidth=2, label=ch)
    ax.fill_between(x_range, y_kde, alpha=0.12, color=color)

    median_val = np.median(values)
    ax.axvline(median_val, color=color, linestyle=':', linewidth=1.4)
    ax.text(median_val, ax.get_ylim()[1] * 0.02, f'{median_val:.1f}%',
            color=color, fontsize=8, ha='center')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('OTP Completion Rate Distribution by Channel', fontsize=14, fontweight='bold')
ax.set_xlabel('OTP Completion Rate (%)')
ax.set_ylabel('Density')
ax.legend(frameon=False, title='Channel')
plt.tight_layout()
plt.show()


---
## Bonus — Correlation heatmap of funnel steps per channel

Build one heatmap per channel showing how correlated the funnel steps are within that channel.

Use `GridSpec` to place 5 heatmaps (one per channel) in a single figure.

What do you notice? Are correlations stronger in organic vs paid?

In [ ]:

# Bonus — Correlation heatmap of funnel steps per channel

import matplotlib.gridspec as gridspec

funnel_steps = [
    'visita_landing', 'inicio_solicitud', 'datos_personales', 'otp',
    'datos_financieros', 'carga_documentos', 'evaluacion_crediticia',
    'aprobacion', 'firma_digital', 'activacion_tarjeta'
]
step_labels = ['Visit', 'Start', 'Personal', 'OTP', 'Financial',
               'Docs', 'Credit', 'Approved', 'Signed', 'Activated']

channels = df['channel'].unique()
n = len(channels)

fig = plt.figure(figsize=(5 * n, 4.5))
gs  = gridspec.GridSpec(1, n, figure=fig, wspace=0.35)

for i, ch in enumerate(channels):
    corr = df[df['channel'] == ch][funnel_steps].corr()
    ax = fig.add_subplot(gs[i])
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(len(step_labels)))
    ax.set_yticks(range(len(step_labels)))
    ax.set_xticklabels(step_labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(step_labels, fontsize=7)
    ax.set_title(ch.capitalize(), fontsize=12, fontweight='bold', pad=8)
    for row in range(len(funnel_steps)):
        for col in range(len(funnel_steps)):
            ax.text(col, row, f'{corr.iloc[row, col]:.2f}',
                    ha='center', va='center', fontsize=5.5, color='black')

fig.colorbar(im, ax=fig.axes, shrink=0.6, label='Pearson r')
fig.suptitle('Funnel Step Correlations by Channel', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Observation: organic tends to show stronger correlations across all funnel steps
# (users follow a more consistent path), while paid traffic shows weaker mid-funnel
# correlations, suggesting more variable behavior after landing.
